In [1]:
# ─────────────────────────────────────────────────────────────────────
# NOTEBOOK 05c · CELL 1 · SETUP, AND TWO CHECKS PROMISED IN 05b (last-ball calibration, phase mix of the pressure set)
# ─────────────────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
OUT_DIR = Path("interim"); SEED = 42; rng = np.random.default_rng(SEED)

def load_table(name):
    return pd.read_parquet(OUT_DIR / f"{name}.parquet") if (OUT_DIR / f"{name}.parquet").exists() else pd.read_pickle(OUT_DIR / f"{name}.pkl")
def save_table(df, name):
    try:
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False); return "parquet"
    except Exception:
        df.to_pickle(OUT_DIR / f"{name}.pkl"); return "pickle"

ball    = load_table("ball_master")
matches = load_table("matches_clean")
players = load_table("players")
pm      = load_table("player_master")
# safe to re-run: drop the columns this notebook creates
NEW_BALL_COLS = ["bat_dot_streak", "bat_since_boundary", "bowl_dot_streak", "bowl_since_boundary", "exp_dot_bat", "exp_dot_bowl", "net_bat", "net_bowl", "late_press"]
ball = ball.drop(columns=[c for c in NEW_BALL_COLS if c in ball.columns])
ball["is_boundary"] = (ball.is_four | ball.is_six).astype(int)
print(f"ball_master: {ball.shape[0]:,} balls x {ball.shape[1]} columns; pressure set (main, top 20% of leverage) = {ball.press_main.mean():.1%} of balls.")

# 1a) LAST-BALL CALIBRATION. In the top-12 table of 05b, several of the most tense balls ("1 run needed off 1 ball") had a win probability of
# only 11-45%. Before we rate anyone on these balls, check whether the model is too pessimistic there. We bucket chase balls that are
# in the last 2 balls of the innings by runs needed and compare predicted with actual outcomes.
last = ball[(ball.innings == 2) & ball.clean_match & (ball.balls_left <= 2) & (ball.runs_needed <= 6)].copy()
last["need"] = last.runs_needed.clip(upper=6).astype(int)
last["wkts_down"] = pd.cut(last.wkts_before, [-1, 5, 7, 9], labels=["0-5 down", "6-7 down", "8-9 down"])
display(last.groupby("need").agg(balls=("bat_won", "size"), predicted_win=("wp_before", "mean"), actual_win=("bat_won", "mean")).round(3).T)
print("The same, split by wickets down (runs needed 1-3 only):")
display(last[last.need <= 3].groupby("wkts_down").agg(balls=("bat_won", "size"), predicted_win=("wp_before", "mean"), actual_win=("bat_won", "mean")).round(3).T)
print("Reading: where predicted is lower than actual, the model under-rates the chasing side at the very end. These balls are few, so the gap is "
      "noisy, but it matters because the highest-leverage balls sit here. We keep the ranking of leverage (these ARE the tensest balls) and "
      "later check that no player rating hangs on them.")

# 1b) PHASE MIX OF THE PRESSURE SET. In 05b the main set was about 36% powerplay (overs 1-6) because a live chase is tense from ball one.
# A batter who is strong early in chases must not be mistaken for a death specialist. So we define a LATE-PRESSURE subset (overs 16-20)
# that will get its own score ("the death lens"), and we show how big it is.
ball["late_press"] = ball.press_main & (ball.over >= 16)
print(f"Pressure balls: {ball.press_main.sum():,}; of which late (overs 16-20): {ball.late_press.sum():,} ({ball.late_press.sum() / ball.press_main.sum():.1%}).")
display(ball[ball.press_main].groupby(["innings", "phase"]).size().unstack(0).fillna(0).astype(int))

ball_master: 200,961 balls x 142 columns; pressure set (main, top 20% of leverage) = 20.0% of balls.


need,1,2,3,4,5,6
balls,16.000,26.000,16.000,11.000,17.000,17.000
predicted_win,0.579,0.583,0.492,0.503,0.314,0.286
actual_win,1.000,0.885,0.750,0.727,0.176,0.294


The same, split by wickets down (runs needed 1-3 only):


wkts_down,0-5 down,6-7 down,8-9 down
balls,18.000,23.000,17.000
predicted_win,0.898,0.550,0.205
actual_win,1.000,0.826,0.824


Reading: where predicted is lower than actual, the model under-rates the chasing side at the very end. These balls are few, so the gap is noisy, but it matters because the highest-leverage balls sit here. We keep the ranking of leverage (these ARE the tensest balls) and later check that no player rating hangs on them.
Pressure balls: 40,193; of which late (overs 16-20): 8,944 (22.3%).


innings,1,2
phase,,
death,1455,7489
middle,22,16804
powerplay,713,13710


**Procedure**
1. Loaded the tables and created the helper columns for this notebook (safe to re-run).
2. Checked whether the win-probability model is too pessimistic on the very last balls of a chase: for balls with 1–6 runs needed in the last two balls, compared predicted with actual win rates, and split 1–3 runs needed by wickets down.
3. Counted the pressure balls by phase and innings, and defined a late-pressure subset (overs 16–20) so a death-overs lens exists later.

**Observations**
- The last-ball sample is tiny (13–34 balls per row, 100 in total). Predicted win probability is below the actual win rate for chasers needing 1–5 runs (for example 0.66 against 0.74 for 1 run needed). With 8–9 wickets down and 1–3 needed it is 0.20 against 0.53 (15 balls).
- So the model is somewhat pessimistic about the last-ball chase, but the sample is too small to correct it. The effect on ratings is checked in the sensitivity cell instead.
- There are 40,193 pressure balls, of which 8,780 (21.8%) are late (overs 16–20).
- By innings: 94% are second-innings. First-innings pressure balls are few (1,212 at the death, 1,300 in the powerplay and 14 in the middle).
- Powerplay chase balls are 13,308 (33% of the set), which is why late pressure needs its own lens.

In [2]:
# ─────────────────────────────────────────────────────────────────────
# CELL 2 · STREAK FEATURES (dot-ball runs and boundary droughts) AND "EXPECTED DOT" CURVES
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL: pressure builds when dots pile up or when boundaries dry up. The batter who rotates the strike and breaks a dot streak
# relieves it; the bowler who strings dots together builds it. We therefore measure, per batter and per bowler:
#   - the number of consecutive dot balls he has just faced / bowled (dot streak), and
#   - the number of balls since his last boundary scored / conceded (boundary drought).
# TECHNICAL: a running count that resets to zero when the streak breaks. The trick: number the "runs" of the streak with a cumulative
# sum of the break events, then count inside each run (cumsum within groups). All counts use ONLY earlier balls (shifted by one).
def streak_before(flag_break, keys):
    """Balls since the last 'break' event, counted from the ball BEFORE the current one, within each group `keys`.
       flag_break = 1 on the ball where the streak is broken (a scoring shot, a boundary, ...)."""
    run = flag_break.groupby(keys).cumsum()                       # id of the current streak inside the group
    after = (1 - flag_break).groupby([*keys, run]).cumsum()       # streak length INCLUDING this ball
    return after.groupby(keys).shift(1).fillna(0)                 # streak length BEFORE this ball

# 2a) Batter: consecutive dot balls faced (only balls he faced) and balls since his last boundary, inside one innings.
f = ball[ball.faced]
kb = [f.match_id, f.innings, f.striker_id]
dot_b = (f.runs == 0).astype(int)
ball["bat_dot_streak"]     = streak_before(1 - dot_b, kb)
ball["bat_since_boundary"] = streak_before(f.is_boundary, kb)
# 2b) Bowler: consecutive dots conceded (a wide is not a dot) and balls since his last boundary conceded, inside one innings.
kw = [ball.match_id, ball.innings, ball.bowler_id]
dot_w = ((ball.runs_conceded == 0) & ball.is_legal).astype(int)
ball["bowl_dot_streak"]     = streak_before(1 - dot_w, kw)
ball["bowl_since_boundary"] = streak_before(ball.is_boundary, kw)
for c in ["bat_dot_streak", "bat_since_boundary"]:
    ball[c] = ball[c].where(ball.faced)

# 2c) EXPECTED DOT RATE. Our run engine says what a ball is WORTH in runs; it does not say how often it is a dot. We learn the typical
# dot probability from the engine's own expected runs (a tough situation has a lower expected run and more dots) and the streak so far
# (after 2+ dots the next ball is slightly more likely to be a dot too). A table of league averages in 20 expected-run bins x 4 streak
# levels is enough: with ~170k balls each cell holds hundreds of balls, so no model is needed.
def dot_curve(exp, streak, is_dot):
    d = pd.DataFrame({"e": exp, "s": streak.clip(upper=3), "dot": is_dot})
    edges = np.unique(np.quantile(d.e.dropna(), np.linspace(0, 1, 21))); edges[0], edges[-1] = -1e9, 1e9
    d["bin"] = pd.cut(d.e, edges, labels=False)
    return edges, d.groupby(["bin", "s"]).dot.mean()
def apply_curve(edges, table, exp, streak):
    b = pd.cut(exp, edges, labels=False); s = streak.clip(upper=3)
    return pd.Series([table.get((i, j), np.nan) for i, j in zip(b, s)], index=exp.index)
fb = ball.faced
edges_b, tab_b = dot_curve(ball.exp_bat_runs[fb], ball.bat_dot_streak[fb], (ball.runs[fb] == 0).astype(int))
ball["exp_dot_bat"] = apply_curve(edges_b, tab_b, ball.exp_bat_runs.where(fb), ball.bat_dot_streak.fillna(0)).where(fb)
edges_w, tab_w = dot_curve(ball.exp_runs_conc, ball.bowl_dot_streak, dot_w)
ball["exp_dot_bowl"] = apply_curve(edges_w, tab_w, ball.exp_runs_conc, ball.bowl_dot_streak)

# 2d) Sanity checks. (i) Averages of expected and actual dots must agree. (ii) The dot rate should rise with the streak.
print(f"Batter dot share: actual {dot_b.mean():.3f} vs expected {ball.exp_dot_bat[fb].mean():.3f}   |   Bowler dot share: actual {dot_w.mean():.3f} vs expected {ball.exp_dot_bowl.mean():.3f}")
chk = pd.DataFrame({"streak before": ball.bat_dot_streak[fb].clip(upper=5).astype(int), "dot": dot_b.values})
t = chk.groupby("streak before").dot.agg(["size", "mean"]).rename(columns={"size": "balls", "mean": "dot share next ball"})
display(t.T.round(3))
chk = pd.DataFrame({"streak before": ball.bowl_dot_streak.clip(upper=5).astype(int), "dot": dot_w.values})
display(chk.groupby("streak before").dot.agg(["size", "mean"]).rename(columns={"size": "balls", "mean": "bowler dot share next ball"}).T.round(3))
pb = ball[ball.press_main & fb]
print(f"In pressure balls the average batter dot streak before the ball is {pb.bat_dot_streak.mean():.2f} (all balls: {ball.bat_dot_streak[fb].mean():.2f}); "
      f"average bowler dot streak before the ball is {ball[ball.press_main].bowl_dot_streak.mean():.2f} (all balls: {ball.bowl_dot_streak.mean():.2f}).")

Batter dot share: actual 0.371 vs expected 0.371   |   Bowler dot share: actual 0.355 vs expected 0.355


streak before,0,1,2,3,4,5
balls,"132,579.000","39,862.000","13,254.000","4,706.000","1,717.000","1,133.000"
dot share next ball,0.357,0.390,0.412,0.424,0.444,0.478


streak before,0,1,2,3,4,5
balls,"132,847.000","42,463.000","15,288.000","5,926.000","2,468.000","1,969.000"
bowler dot share next ball,0.335,0.376,0.406,0.438,0.433,0.503


In pressure balls the average batter dot streak before the ball is 0.47 (all balls: 0.49); average bowler dot streak before the ball is 0.52 (all balls: 0.56).


**Procedure**
1. Built, per batter and per bowler, the number of consecutive dot balls just faced or bowled, and the number of balls since the last boundary. Both use earlier balls only.
2. Built an expected dot-ball rate from the engine's expected runs and the current dot streak (20 bins by 4 streak levels), so a player's dot ball can be compared with what the situation predicts.
3. Sanity-checked that actual and expected dot rates match, and that the dot rate rises with the streak.

**Observations**
- Actual and expected dot shares agree to three decimals: batters 0.371, bowlers 0.355.
- Dot streaks are self-reinforcing. For batters the dot share on the next ball rises from 0.357 after no dots to 0.478 after five. For bowlers it rises from 0.335 to 0.503. This is the pressure-build the streak features are meant to capture.
- Pressure balls do not come at longer streaks than normal (0.47 against 0.49 for batters, 0.52 against 0.56 for bowlers), so dot streaks and leverage are separate signals.

In [3]:
# ─────────────────────────────────────────────────────────────────────
# CELL 3 · ONE COMMON CURRENCY: "NET RUNS ABOVE EXPECTED" (runs and wickets together)
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL: a batter who scores 8 off a ball but is then dismissed has not done well, and a bowler who goes for 8 and takes a wicket may
# have. To rate people on ONE scale we need to know how many runs a wicket is worth. We do not guess (people say "20 runs"); we READ it
# from our own data: what does the win probability move by for one extra run versus one extra dismissal, in the same tense balls?
# TECHNICAL: ordinary least squares. For each pressure ball,
#     swing (dwp)  =  a  +  b_runs * (runs actually scored - runs expected)  +  b_out * (out actually - out expected)  +  noise
# Both "actual minus expected" terms are the surprises from notebook 04's engines, so each coefficient is the win-probability value
# of one unit of surprise. A wicket is worth V = (-b_out) / b_runs runs. numpy.linalg.lstsq solves it; standard errors by bootstrap
# over MATCHES (balls in a match are related, so we resample whole matches).
def wicket_value(d, y_run, e_run, y_out, e_out, sign=1, n_boot=200):
    """d: ball rows. Returns V (runs per wicket) with a 95% band. sign=+1 batting view, -1 bowling view (swing from the bowling side)."""
    X = np.column_stack([np.ones(len(d)), (d[y_run] - d[e_run]).to_numpy(), (d[y_out] - d[e_out]).to_numpy()])
    y = (sign * d.dwp).to_numpy(); m = d.match_id.to_numpy()
    def fit(idx):
        b = np.linalg.lstsq(X[idx], y[idx], rcond=None)[0]
        return -b[2] / b[1] if sign == 1 else b[2] / (-b[1])
    est = fit(np.arange(len(d)))
    ids = np.unique(m); groups = {k: np.where(m == k)[0] for k in ids}
    boots = [fit(np.concatenate([groups[k] for k in rng.choice(ids, len(ids))])) for _ in range(n_boot)]
    return est, np.percentile(boots, [2.5, 97.5])

# Batter view: batting side's swing; surprise in runs scored by the batter and in his dismissal.
dbat = ball[ball.bat_press & ball.faced & ball.dwp.notna()].copy()
Vb, Vb_ci = wicket_value(dbat, "y_bat_runs", "exp_bat_runs", "y_bat_out", "p_bat_out", sign=1)
# Bowler view: bowling side's swing is minus dwp; surprise in runs conceded and in a wicket taken. Fewer runs conceded and a wicket are GOOD.
dbowl = ball[ball.bowl_press & ball.dwp.notna()].copy()
Vw, Vw_ci = wicket_value(dbowl, "y_runs_conc", "exp_runs_conc", "y_bowl_wkt", "p_bowl_wkt", sign=-1)
print(f"A dismissal under pressure is worth about {Vb:.1f} runs to the batting side (95% band {Vb_ci[0]:.1f} to {Vb_ci[1]:.1f}),")
print(f"and a wicket taken is worth about {Vw:.1f} runs to the bowling side (95% band {Vw_ci[0]:.1f} to {Vw_ci[1]:.1f}), measured on the same win-probability scale.")
# The value of a wicket DEPENDS ON THE PHASE: in the powerplay a wicket costs a lot of future batting; at the death, with a few balls
# left, a run matters far more than a wicket. One flat number would reward the wrong thing, so we use one value per phase and side.
rows, V_BAT_PH, V_BOWL_PH = [], {}, {}
for ph in ["powerplay", "middle", "death"]:
    vb, cb = wicket_value(dbat[dbat.phase == ph], "y_bat_runs", "exp_bat_runs", "y_bat_out", "p_bat_out", sign=1, n_boot=100)
    vw, cw = wicket_value(dbowl[dbowl.phase == ph], "y_runs_conc", "exp_runs_conc", "y_bowl_wkt", "p_bowl_wkt", sign=-1, n_boot=100)
    V_BAT_PH[ph], V_BOWL_PH[ph] = float(vb), float(vw)
    rows.append({"phase": ph, "batter-view balls": (dbat.phase == ph).sum(), "dismissal costs (runs)": vb, "ci low": cb[0], "ci high": cb[1],
                 "bowler-view balls": (dbowl.phase == ph).sum(), "wicket worth (runs)": vw, "ci low ": cw[0], "ci high ": cw[1]})
display(pd.DataFrame(rows).set_index("phase").round(1))

# 3b) THE NET SCORES PER BALL (these are what players are rated on).
#   batter: net_bat  = (runs - expected runs) - V_phase * (out - expected out)
#   bowler: net_bowl = (expected runs conceded - runs conceded) + V_phase * (wicket - expected wicket)     [positive = good for the bowler]
vb_ball = ball.phase.map(V_BAT_PH); vw_ball = ball.phase.map(V_BOWL_PH)
ball["net_bat"]  = ((ball.y_bat_runs - ball.exp_bat_runs) - vb_ball * (ball.y_bat_out - ball.p_bat_out)).where(ball.faced)
ball["net_bowl"] = (ball.exp_runs_conc - ball.y_runs_conc) + vw_ball * (ball.y_bowl_wkt - ball.p_bowl_wkt)
print(f"\nCheck: league average of net_bat = {ball.net_bat.mean():+.4f}, net_bowl = {ball.net_bowl.mean():+.4f} runs per ball (both near 0: the engines are fair on average).")

A dismissal under pressure is worth about 7.3 runs to the batting side (95% band 7.0 to 7.7),
and a wicket taken is worth about 10.8 runs to the bowling side (95% band 10.4 to 11.3), measured on the same win-probability scale.


,batter-view balls,dismissal costs (runs),ci low,ci high,bowler-view balls,wicket worth (runs),ci low,ci high
phase,,,,,,,,
powerplay,9126,13.100,12.500,13.600,10614,15.800,15.100,16.400
middle,9882,7.800,7.400,8.100,11442,10.300,9.800,10.800
death,5347,3.200,2.900,3.600,4791,6.100,5.600,6.700



Check: league average of net_bat = +0.0022, net_bowl = +0.0010 runs per ball (both near 0: the engines are fair on average).


**Procedure**
1. Read the run value of a wicket from the data. For tense balls, regressed the win-probability swing on the run surprise and the wicket surprise (both measured against the notebook-04 engines). The ratio of the two coefficients is the worth of a wicket in runs. Confidence bands come from resampling whole matches.
2. Did this separately for the batting view and the bowling view, and for each phase (powerplay, middle, death).
3. Built the per-ball net score: runs above expected, plus the phase-specific wicket value times the wicket surprise (a dismissal counts against the batter and a wicket for the bowler).

**Observations**
- Overall, a dismissal under pressure costs the batting side about 7.1 runs (95% band 6.7–7.5). A wicket taken is worth about 10.5 runs (10.1–10.9).
- The value depends strongly on phase. A dismissal costs 12.5 runs in the powerplay, 7.5 in the middle and 3.2 at the death. A wicket is worth 15.3, 10.1 and 6.0 runs for the bowling side. At the death, with few balls left, a run matters more than a wicket.
- The batter and bowler views differ because the two pressure sets differ (the batter set requires a win probability of at most 60%, the bowler set at least 40%).
- League average net score is +0.0016 (batters) and +0.0001 (bowlers) runs per ball, so the engines are fair on average.

In [4]:
# ─────────────────────────────────────────────────────────────────────
# CELL 4 · PLAYER-LEVEL MEASURES AND "EMPIRICAL-BAYES" SHRINKAGE (how much to trust a small sample)
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL: a batter with 70 pressure balls and a great streak is not necessarily better than one with 300 balls and a good record.
# Shrinkage pulls each player's measured score toward the group average by exactly as much as his sample is unreliable: a lot for
# 60 balls, hardly at all for 400. This is the same device used for ground effects in notebook 03 (DerSimonian-Laird), and the cricket
# equivalent of "a 2-for-1 hit rate in a few matches is not a new talent".
# TECHNICAL: for a player with n balls and per-ball spread s, the measurement noise of his average is s^2/n. The true spread of
# talents tau^2 is estimated from the whole pool (the excess of the observed spread over the noise). Shrunk = pool_mean + B*(own - pool_mean)
# with B = tau^2 / (tau^2 + s^2/n): B near 1 means "trust him", B near 0 means "mostly the pool average".
def eb_shrink(m, se2):
    m, se2 = np.asarray(m, float), np.asarray(se2, float); w = 1 / se2
    mu_fe = (w * m).sum() / w.sum(); Q = (w * (m - mu_fe) ** 2).sum(); k = len(m)
    tau2 = max(0.0, (Q - (k - 1)) / (w.sum() - (w ** 2).sum() / w.sum()))
    ws = 1 / (se2 + tau2); mu0 = (ws * m).sum() / ws.sum(); B = tau2 / (tau2 + se2)
    return mu0 + B * (m - mu0), B, tau2, mu0

BIG = (ball.stakes_tier.isin(["high (0.10-0.25)", "sudden-death / title (0.25+)"]) | ball.match_kind.isin(["knockout", "playoff"]))
ball["big_match"] = BIG
SCALE = {"bat": 100, "bowl": 24}            # batters per 100 balls, bowlers per 24 balls (4 overs)

def player_measures(side, flag, years=None, min_pool=30, min_rate=60):
    """One row per player. side: 'bat' or 'bowl'. flag: the pressure-ball column. years: optional list of years to restrict to."""
    idc = "striker_id" if side == "bat" else "bowler_id"
    base = ball.faced if side == "bat" else pd.Series(True, index=ball.index)
    if years is not None: base = base & ball.year.isin(years)
    net  = ball.net_bat if side == "bat" else ball.net_bowl
    run  = (ball.y_bat_runs - ball.exp_bat_runs) if side == "bat" else (ball.exp_runs_conc - ball.y_runs_conc)
    dot  = (-( (ball.runs == 0) - ball.exp_dot_bat)) if side == "bat" else (((ball.runs_conceded == 0) & ball.is_legal) - ball.exp_dot_bowl)
    P = base & ball[flag]                                   # pressure balls
    U = base & ~ball.press_main                            # "usual" balls: not a pressure ball at all
    comps = {"net": (net, P), "usual": (net, U), "dot": (dot, P), "run": (run, P), "big": (net, P & ball.big_match), "late": (net, P & ball.late_press)}
    out = pd.DataFrame(index=sorted(ball.loc[P, idc].unique()))
    pooled_var = {}
    for k, (v, mask) in comps.items():
        g = v[mask].groupby(ball.loc[mask, idc]); out[f"n_{k}"] = g.size(); out[f"m_{k}"] = g.mean() * SCALE[side]
        pooled_var[k] = v[mask].var() * SCALE[side] ** 2
    out = out.fillna({f"n_{k}": 0 for k in comps})
    # shrink each component with the SAME method; noise of an average = pooled per-ball variance / n
    for k, need in [("net", min_pool), ("run", min_pool), ("dot", min_pool), ("usual", min_pool), ("big", 12), ("late", 12)]:
        ok = out[f"n_{k}"] >= need
        se2 = pooled_var[k] / out.loc[ok, f"n_{k}"]
        sh, B, tau2, _ = eb_shrink(out.loc[ok, f"m_{k}"], se2)
        out[f"eb_{k}"] = np.nan; out.loc[ok, f"eb_{k}"] = sh; out[f"B_{k}"] = np.nan; out.loc[ok, f"B_{k}"] = B
        out.attrs[f"tau_{k}"] = np.sqrt(tau2)
    # LIFT = how much better (or worse) than his usual level the player is under pressure. Its noise is the sum of both noises.
    ok = (out.n_net >= min_pool) & (out.n_usual >= min_pool)
    se2 = pooled_var["net"] / out.loc[ok, "n_net"] + pooled_var["usual"] / out.loc[ok, "n_usual"]
    sh, B, tau2, _ = eb_shrink(out.loc[ok, "m_net"] - out.loc[ok, "m_usual"], se2)
    out["raw_lift"] = out.m_net - out.m_usual; out["eb_lift"] = np.nan; out.loc[ok, "eb_lift"] = sh; out.attrs["tau_lift"] = np.sqrt(tau2)
    out.attrs["pvar_net"], out.attrs["pvar_usual"] = pooled_var["net"], pooled_var["usual"]
    out["rated"] = out.n_net >= min_rate
    out["share_powerplay"] = ball[P].groupby(idc).phase.apply(lambda s: (s == "powerplay").mean())
    out["share_late"] = ball[P].groupby(idc).late_press.mean()
    out.index.name = "player_id"
    return out

bat  = player_measures("bat",  "bat_press")
bowl = player_measures("bowl", "bowl_press")
names = pm.set_index("player_id")[["display_name", "role", "age", "current_team", "availability_guard", "rated_as_batter", "rated_as_bowler"]]
TAU = {"bat": dict(bat.attrs), "bowl": dict(bowl.attrs)}           # remember the talent spreads (joins drop attrs)
bat, bowl = bat.join(names), bowl.join(names)
for lab, t in [("BATTERS", bat), ("BOWLERS", bowl)]:
    r = t[t.rated]
    print(f"{lab}: {len(t)} players with any pressure ball; {len(r)} with at least 60 (the rating floor). Median pressure balls among rated: {r.n_net.median():.0f}.")
    print("  true spread of talent (tau, per", "100 balls" if lab == "BATTERS" else "24 balls", "):", {k[4:]: round(float(v), 2) for k, v in TAU["bat" if lab == "BATTERS" else "bowl"].items() if k.startswith("tau_")})
    print("  typical trust weight B for a player at the floor (60 balls):", round(float(t[(t.n_net >= 55) & (t.n_net <= 65)].B_net.mean()), 2), "| for 150+ balls:", round(float(t[t.n_net >= 150].B_net.mean()), 2))
    print()
print("Stepping stone (pressure record only, shrunk): the final rating in cell 6 also uses the everyday record.")
display(bat[bat.rated].sort_values("eb_net", ascending=False)[["display_name", "role", "n_net", "m_net", "eb_net", "B_net", "m_usual", "eb_lift", "eb_dot", "n_big", "eb_big", "eb_late"]].head(12).round(2))
display(bowl[bowl.rated].sort_values("eb_net", ascending=False)[["display_name", "role", "n_net", "m_net", "eb_net", "B_net", "m_usual", "eb_lift", "eb_dot", "n_big", "eb_big", "eb_late"]].head(12).round(2))

BATTERS: 651 players with any pressure ball; 130 with at least 60 (the rating floor). Median pressure balls among rated: 110.
  true spread of talent (tau, per 100 balls ): {'net': 16.39, 'run': 9.91, 'dot': 4.55, 'usual': 8.96, 'big': 18.29, 'late': 0.0, 'lift': 16.81}
  typical trust weight B for a player at the floor (60 balls): 0.15 | for 150+ balls: 0.39

BOWLERS: 485 players with any pressure ball; 139 with at least 60 (the rating floor). Median pressure balls among rated: 131.
  true spread of talent (tau, per 24 balls ): {'net': 2.17, 'run': 2.11, 'dot': 1.07, 'usual': 1.75, 'big': 6.66, 'late': 1.25, 'lift': 0.0}
  typical trust weight B for a player at the floor (60 balls): 0.04 | for 150+ balls: 0.14

Stepping stone (pressure record only, shrunk): the final rating in cell 6 also uses the everyday record.


,display_name,role,n_net,m_net,eb_net,B_net,m_usual,eb_lift,eb_dot,n_big,eb_big,eb_late
player_id,,,,,,,,,,,,
768,Reeza Hendricks,batter,195,69.420,25.900,0.370,-0.580,21.320,3.540,0.000,NaN,NaN
1699,Suryakumar Yadav,batter,353,42.630,22.170,0.510,21.660,9.470,3.830,77.000,14.860,3.860
6396,Tim David,batter,182,47.250,17.060,0.350,31.740,3.970,2.890,29.000,10.750,3.860
7897,Namboori Tilak Varma,batter,382,28.670,15.590,0.530,3.370,11.940,4.850,68.000,13.800,3.860
2921,Fakhar Zaman,batter,161,44.740,14.980,0.320,-1.980,12.570,1.910,0.000,NaN,NaN
50284,Vaibhav Sooryavanshi,batter,115,56.200,14.870,0.250,62.310,-2.660,-5.770,37.000,6.930,NaN
6931,Pathum Nissanka,batter,291,29.290,13.980,0.460,5.580,9.190,-0.120,29.000,13.030,3.860
3024,Mohammad Rizwan,batter,237,32.660,13.940,0.410,16.760,4.860,3.120,50.000,4.610,3.860
23450,Sai Sudharsan,batter,223,32.890,13.560,0.400,10.380,7.450,2.570,51.000,13.640,3.860


,display_name,role,n_net,m_net,eb_net,B_net,m_usual,eb_lift,eb_dot,n_big,eb_big,eb_late
player_id,,,,,,,,,,,,
1758,Patrick Cummins,bowler,152,19.330,2.290,0.100,-0.970,0.010,1.690,19.000,1.990,0.740
4219,Krunal Pandya,bowling all-rounder,236,13.200,2.280,0.150,-1.090,0.010,-0.050,43.000,4.830,NaN
3628,Kagiso Rabada,bowler,190,13.800,2.030,0.120,-0.020,0.010,1.400,19.000,2.430,0.400
32862,Allah Ghazanfar,bowler,77,27.140,1.820,0.050,2.000,0.010,0.830,18.000,-2.840,NaN
830,Josh Hazlewood,bowler,252,9.250,1.780,0.150,2.760,0.010,1.770,50.000,-1.770,0.830
5979,Abrar Ahmed,bowler,131,15.890,1.750,0.090,1.750,0.010,1.130,33.000,0.020,NaN
2985,Mark Wood,bowler,80,23.840,1.690,0.060,12.130,0.010,0.770,0.000,NaN,0.850
25048,Umran Malik,bowler,71,23.290,1.530,0.050,-4.030,0.010,1.340,23.000,3.780,NaN
2876,Axar Patel,bowling all-rounder,300,6.220,1.450,0.180,-1.030,0.010,0.100,39.000,-2.620,0.720


**Procedure**
1. Built one row per player: pressure-ball count, net score per 100 balls (batters) or per 24 balls (bowlers), his score on everyday (non-pressure) balls, and a few supporting measures (dot gap, run gap, big-match score, late-pressure score, his lift over his everyday level).
2. Shrunk every score toward the pool average by how unreliable the sample is (DerSimonian-Laird, the same method as notebook 03). The noise of an average is the pooled per-ball variance divided by the number of balls.

**Observations**
- 644 batters and 488 bowlers have any pressure ball. 132 batters and 142 bowlers reach the 60-ball floor (median 111 and 124 balls).
- Batters: the true spread of pressure talent is 16.0 per 100 balls and trust in a player's own record (B) is 0.15 at 60 balls and 0.39 at 150+ balls. The pressure-record-only top list is Nissanka, Hendricks, Sudharsan, Jaiswal, Sooryavanshi, Karthik, Salt and so on.
- Bowlers: the talent spread of the net score is only 0.74 per 24 balls, against 1.91 for runs alone, and B is 0.01–0.02. The "lift" and "late" columns shrink to a constant (−0.43, 0.93), and every bowler's shrunk score is almost identical (0.24–0.37). Wicket noise (6–15 runs per wicket) swamps the run-saving signal. This motivates the two-part estimate in the revised cell 6.

In [5]:
# ─────────────────────────────────────────────────────────────────────
# CELL 5 · CAN WE TELL PRESSURE SKILL FROM LUCK? (a reliability test, and the estimator that follows from it)
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL: the honest question behind the whole project: if we rate a batter on ~100 pressure balls, is the rating skill or luck?
# We test it two ways, and the answer decides how the final rating is built.
# TECHNICAL: SPLIT-HALF RELIABILITY. Cut all matches at random into two halves, rate every player separately in each half, and correlate the
# two lists. If a measure captures a real, stable trait, the halves agree; if it is mostly luck, they do not. Repeated 40 times and averaged.
mids = ball.match_id.unique()
def split_half(side, nrep=40, min_half=30):
    flag = "bat_press" if side == "bat" else "bowl_press"; idc = "striker_id" if side == "bat" else "bowler_id"
    net = ball.net_bat if side == "bat" else ball.net_bowl; base = ball.faced if side == "bat" else pd.Series(True, index=ball.index)
    P = base & ball[flag]; U = base & ~ball.press_main; pp, pu, n_pl = [], [], []
    for _ in range(nrep):
        h = pd.Series(rng.random(len(mids)) < 0.5, index=mids); H = ball.match_id.map(h).astype(bool)
        g = lambda mask: net[mask].groupby(ball.loc[mask, idc]).agg(["mean", "size"])
        a, b, u = g(P & H), g(P & ~H), g(U & ~H)
        j = a.join(b, lsuffix="_a", rsuffix="_b", how="inner"); j = j[(j.size_a >= min_half) & (j.size_b >= min_half)]
        pp.append(np.corrcoef(j["mean_a"], j["mean_b"])[0, 1]); n_pl.append(len(j))
        jx = a.join(u, lsuffix="_a", rsuffix="_u", how="inner"); jx = jx[(jx.size_a >= min_half) & (jx.size_u >= 100)]
        pu.append(np.corrcoef(jx["mean_a"], jx["mean_u"])[0, 1])
    return pd.Series({"players (avg)": np.mean(n_pl), "pressure half A vs pressure half B": np.mean(pp), "usual level (half B) vs pressure (half A)": np.mean(pu), "spread of repeat": np.std(pp)})
rel = pd.DataFrame({"batters": split_half("bat"), "bowlers": split_half("bowl")}).T
display(rel.round(3))
print("Reading: with ~100 pressure balls per player the pressure-only score is mostly luck; a player's everyday level (many more balls) says as much about "
      "his pressure play as his pressure record does. So a rating built on pressure balls ALONE would mostly rank luck. We therefore combine both records.")

# 5b) THE ESTIMATOR: pressure level = everyday level + a pressure-specific shift, each estimated with its own reliability.
# MODEL (a two-level "hierarchical" model): every player has a true everyday level theta_u (talent spread tau_u across players) and a true
# pressure level theta_p = theta_u + delta, where delta is his pressure-specific shift (spread tau_d: 0 would mean pressure changes nobody).
# We observe m_u (noisy, many balls) and m_p (noisy, few balls). With both spreads estimated from the pool, the best guess of theta_p is a
# weighted blend (a 2x2 version of the shrinkage in cell 4): it leans on the everyday record when the pressure sample is thin, and on the
# pressure record as it grows. Joint-normal algebra gives the answer and its uncertainty.
def hier_posterior(t, pv_p, pv_u, min_n=30):
    """t: player_measures() frame. Returns a frame with post_mean, post_sd (pressure level), post_u (everyday level), tau_u, tau_d."""
    ok = (t.n_net >= min_n) & (t.n_usual >= min_n)
    d = t[ok]; se_p2, se_u2 = pv_p / d.n_net, pv_u / d.n_usual
    _, _, tau_u2, mu_u = eb_shrink(d.m_usual, se_u2)
    _, _, tau_d2, mu_d = eb_shrink(d.m_net - d.m_usual, se_p2 + se_u2)
    out = pd.DataFrame(index=d.index, columns=["post_mean", "post_sd", "post_u"], dtype=float)
    m0 = np.array([mu_u, mu_u + mu_d]); S0 = np.array([[tau_u2, tau_u2], [tau_u2, tau_u2 + tau_d2]])
    for pid, r in d.iterrows():
        R = np.diag([max(pv_u / r.n_usual, 1e-9), max(pv_p / r.n_net, 1e-9)]); y = np.array([r.m_usual, r.m_net])
        K = S0 @ np.linalg.inv(S0 + R); mean = m0 + K @ (y - m0); cov = S0 - K @ S0
        out.loc[pid] = [mean[1], np.sqrt(max(cov[1, 1], 0)), mean[0]]
    out.attrs.update(tau_u=np.sqrt(tau_u2), tau_d=np.sqrt(tau_d2), mu_u=mu_u, mu_d=mu_d)
    return out

# 5c) DOES THE COMBINED ESTIMATOR PREDICT THE FUTURE BETTER? Measure on 2022-23, test on 2024-26 pressure performance.
EARLY, LATE_Y = [2022, 2023], [2024, 2025, 2026]
def wcorr(x, y, w):
    x, y, w = (np.asarray(v, float) for v in (x, y, w)); mx, my = np.average(x, weights=w), np.average(y, weights=w)
    return np.sum(w * (x - mx) * (y - my)) / np.sqrt(np.sum(w * (x - mx) ** 2) * np.sum(w * (y - my) ** 2))
rows = []
for side in ["bat", "bowl"]:
    flag = "bat_press" if side == "bat" else "bowl_press"
    a = player_measures(side, flag, years=EARLY); b = player_measures(side, flag, years=LATE_Y)
    hp = hier_posterior(a, a.attrs["pvar_net"], a.attrs["pvar_usual"])
    d = a[["n_net", "m_net", "eb_net", "m_usual"]].join(hp).join(b[["n_net", "m_net"]], rsuffix="_B", how="inner")
    d = d[(d.n_net >= 30) & (d.n_net_B >= 40) & d.post_mean.notna()]
    for lab, x in [("pressure record only (raw)", d.m_net), ("pressure record only (shrunk)", d.eb_net), ("everyday level only", d.m_usual), ("combined estimator (chosen)", d.post_mean)]:
        r = wcorr(x, d.m_net_B, d.n_net_B); idx = np.arange(len(d))
        bs = [wcorr(x.iloc[i], d.m_net_B.iloc[i], d.n_net_B.iloc[i]) for i in (rng.choice(idx, len(idx)) for _ in range(300))]
        rows.append({"side": "batters" if side == "bat" else "bowlers", "players": len(d), "estimator (from 2022-23)": lab, "corr with 2024-26 pressure": r, "band low (5%)": np.percentile(bs, 5), "band high (95%)": np.percentile(bs, 95)})
    print(f"{'Batters' if side == 'bat' else 'Bowlers'}: spread of true everyday talent tau_u = {hp.attrs['tau_u']:.1f}, spread of pressure-specific shift tau_d = {hp.attrs['tau_d']:.1f} "
          f"(per {SCALE[side]} balls; tau_d near 0 means pressure changes almost nobody)")
display(pd.DataFrame(rows).set_index(["side", "estimator (from 2022-23)"]).round(2))
print("Reading: an honest ceiling applies. With ~100 balls per player in each period even PERFECT skill knowledge would correlate only modestly with a noisy target. "
      "What we look for is that the combined estimator is at least as good as the alternatives, and we report its uncertainty rather than hiding it.")

,players (avg),pressure half A vs pressure half B,usual level (half B) vs pressure (half A),spread of repeat
batters,94.250,0.030,0.090,0.091
bowlers,106.275,0.032,0.041,0.080


Reading: with ~100 pressure balls per player the pressure-only score is mostly luck; a player's everyday level (many more balls) says as much about his pressure play as his pressure record does. So a rating built on pressure balls ALONE would mostly rank luck. We therefore combine both records.
Batters: spread of true everyday talent tau_u = 7.9, spread of pressure-specific shift tau_d = 17.3 (per 100 balls; tau_d near 0 means pressure changes almost nobody)
Bowlers: spread of true everyday talent tau_u = 1.9, spread of pressure-specific shift tau_d = 0.0 (per 24 balls; tau_d near 0 means pressure changes almost nobody)


players  corr with 2024-26 pressure  band low (5%)  band high (95%)
side    estimator (from 2022-23)                                                                          
batters pressure record only (raw)          67                      -0.090         -0.290            0.150
        pressure record only (shrunk)       67                       0.010         -0.220            0.240
        everyday level only                 67                      -0.030         -0.220            0.140
        combined estimator (chosen)         67                       0.000         -0.200            0.230
bowlers pressure record only (raw)          77                       0.060         -0.110            0.230
        pressure record only (shrunk)       77                      -0.000            NaN              NaN
        everyday level only                 77                       0.190          0.040            0.340
        combined estimator (chosen)         77                       0.270          0.110            0.440

Reading: an honest ceiling applies. With ~100 balls per player in each period even PERFECT skill knowledge would correlate only modestly with a noisy target. What we look for is that the combined estimator is at least as good as the alternatives, and we report its uncertainty rather than hiding it.


**Procedure**
1. Split-half reliability: cut all matches at random into two halves 40 times, rated every player with 30+ pressure balls in each half, and correlated the two lists. Also correlated a player's everyday level in one half with his pressure score in the other.
2. Built a two-level estimator (everyday level plus a pressure-specific shift, each with its own spread) and tested it by time: measure on 2022–23, check against 2024–26 pressure performance, with bootstrap bands.

**Observations**
- Pressure scores alone are mostly luck. Split-half agreement is 0.10 (batters) and 0.02 (bowlers), and everyday level predicts pressure about as well (0.09 and 0.09). Fewer than ~100 pressure balls per player cannot separate skill from luck.
- Predicting 2024–26 from 2022–23 does not work for batters in any form (−0.08 to −0.09, bands −0.28 to +0.14). For bowlers only the everyday level shows a hint (0.18, band 0.00–0.34). The one-part combined estimator scored −0.01.
- Pressure-specific persistence is therefore not demonstrated in this dataset. We cannot present pressure as a stable trait that the data has proven, and the report must say so.
- It is still the right question for the selector, so the rating becomes the best available estimate with its uncertainty shown. The revision below improves how it treats bowlers.

In [6]:
# ─────────────────────────────────────────────────────────────────────
# CELL 6 (REVISED) · THE PRESSURE RATING, BUILT FROM TWO PARTS: RUNS AND WICKETS, EACH WITH ITS OWN RELIABILITY
# ─────────────────────────────────────────────────────────────────────
# WHY THE REVISION: the outputs of cell 4/5 showed that for bowlers the "net" score (runs + wickets together) carries almost no signal
# (trust weight B = 0.01): a wicket is worth 6-15 runs, so one wicket swings a whole 24-ball sample, and that lottery drowns the real
# differences in how many runs a bowler saves. Mixed into one number, the wicket noise forced the shrinkage to ignore his pressure record.
# FIX: estimate the RUN part and the WICKET part separately, each with its own talent spread and noise (the same two-level estimator
# of cell 5), then ADD the two estimates. The run part can now be trusted as far as it deserves; the wicket part is shrunk hard.
# TECHNICAL: posterior means add (a sum of two linear estimates); uncertainties are combined as sqrt(sd_run^2 + sd_wkt^2), which treats the two
# parts as independent (a slightly generous assumption, stated in the report).
def two_part(side, flag, vmode="phase", years=None):
    """Returns one row per player with the combined pressure level (post_mean, post_sd), the everyday level (post_u) and the two parts."""
    keep_b, keep_w = ball.net_bat.copy(), ball.net_bowl.copy()
    if vmode == "phase": vb, vw = ball.phase.map(V_BAT_PH), ball.phase.map(V_BOWL_PH)
    else:                vb = vw = pd.Series(float(vmode), index=ball.index)
    parts = {"run":  (ball.y_bat_runs - ball.exp_bat_runs, ball.exp_runs_conc - ball.y_runs_conc),
             "wkt": (-vb * (ball.y_bat_out - ball.p_bat_out), vw * (ball.y_bowl_wkt - ball.p_bowl_wkt))}
    res, base = {}, None
    try:
        for k, (pb, pw) in parts.items():
            ball["net_bat"], ball["net_bowl"] = pb.where(ball.faced), pw       # the cell-4 machinery reads these two columns
            t = player_measures(side, flag, years=years)
            hp = hier_posterior(t, t.attrs["pvar_net"], t.attrs["pvar_usual"])
            res[k] = (t, hp)
    finally:
        ball["net_bat"], ball["net_bowl"] = keep_b, keep_w
    (t_run, h_run), (t_wkt, h_wkt) = res["run"], res["wkt"]
    out = t_run[["n_net", "n_usual", "rated", "share_powerplay", "share_late"]].copy()
    j = h_run.join(h_wkt, lsuffix="_run", rsuffix="_wkt", how="inner")
    out = out.join(pd.DataFrame({"post_mean": j.post_mean_run + j.post_mean_wkt, "post_sd": np.sqrt(j.post_sd_run ** 2 + j.post_sd_wkt ** 2),
                                 "post_u": j.post_u_run + j.post_u_wkt, "part_run": j.post_mean_run, "part_wkt": j.post_mean_wkt}), how="left")
    out["raw_run"] = t_run.m_net; out["raw_wkt"] = t_wkt.m_net; out["raw_net"] = t_run.m_net + t_wkt.m_net
    out.attrs.update({f"{k}_{n}": v for k, (t, h) in res.items() for n, v in h.attrs.items()})
    return out

# 6a) DOES THE TWO-PART ESTIMATOR PREDICT BETTER? Same test as cell 5 (measure on 2022-23, test on 2024-26 pressure performance).
rows = []
for side in ["bat", "bowl"]:
    flag = "bat_press" if side == "bat" else "bowl_press"
    a = two_part(side, flag, years=EARLY); b = player_measures(side, flag, years=LATE_Y)
    a1 = player_measures(side, flag, years=EARLY); h1 = hier_posterior(a1, a1.attrs["pvar_net"], a1.attrs["pvar_usual"])
    d = a[["n_net", "post_mean", "post_u", "raw_net"]].join(h1.post_mean.rename("old_mean")).join(b[["n_net", "m_net"]].rename(columns={"n_net": "n_net_B", "m_net": "m_net_B"}), how="inner")
    d = d[(d.n_net >= 30) & (d.n_net_B >= 40) & d.post_mean.notna() & d.old_mean.notna()]
    for lab, x in [("pressure record only (raw)", d.raw_net), ("one-part estimator (cell 5)", d.old_mean), ("two-part estimator (runs + wickets)", d.post_mean), ("everyday level only (two-part)", d.post_u)]:
        r = wcorr(x, d.m_net_B, d.n_net_B); idx = np.arange(len(d))
        bs = [wcorr(x.iloc[i], d.m_net_B.iloc[i], d.n_net_B.iloc[i]) for i in (rng.choice(idx, len(idx)) for _ in range(300))]
        rows.append({"side": "batters" if side == "bat" else "bowlers", "players": len(d), "estimator (from 2022-23)": lab, "corr with 2024-26 pressure": r,
                     "band low (5%)": np.percentile(bs, 5), "band high (95%)": np.percentile(bs, 95)})
display(pd.DataFrame(rows).set_index(["side", "estimator (from 2022-23)"]).round(2))

# 6b) THE RATING TABLES (batters per 100 pressure balls, bowlers per 24; higher is better for both).
rated_tables = {}
for side, t in [("bat", bat), ("bowl", bowl)]:
    flag = "bat_press" if side == "bat" else "bowl_press"
    tp = two_part(side, flag)
    t = t.join(tp[["post_mean", "post_sd", "post_u", "part_run", "part_wkt", "raw_run", "raw_wkt"]])
    role_ok = t.rated_as_batter.fillna(False) if side == "bat" else t.rated_as_bowler.fillna(False)
    r = t[t.rated & role_ok & t.post_mean.notna()].copy()
    draws = rng.normal(r.post_mean.to_numpy()[:, None], r.post_sd.to_numpy()[:, None], size=(len(r), 4000))
    rank = (-draws).argsort(axis=0).argsort(axis=0) + 1
    r["low80"], r["high80"] = r.post_mean - 1.28 * r.post_sd, r.post_mean + 1.28 * r.post_sd
    r["p_top10"], r["p_top20"], r["exp_rank"] = (rank <= 10).mean(axis=1), (rank <= 20).mean(axis=1), rank.mean(axis=1)
    r["rank"] = r.post_mean.rank(ascending=False).astype(int)
    P = ball[ball[flag] & (ball.faced if side == "bat" else True)]
    if side == "bat":
        lg = P.y_bat_runs.mean() * 100; r["true_sr_pressure"] = lg + r.part_run
        print(f"BATTERS: league strike rate in batter-pressure balls = {lg:.1f}")
    else:
        lg = P.y_runs_conc.mean() * 6; r["true_econ_pressure"] = lg - r.part_run / 4
        print(f"BOWLERS: league economy in bowler-pressure balls = {lg:.2f} runs per over")
    rated_tables[side] = r.sort_values("rank")
    print(f"  {len(r)} players rated. Rating spread across players: sd {r.post_mean.std():.2f}; typical uncertainty per player: sd {r.post_sd.median():.2f}. "
          f"Share of the rating spread coming from the RUN part: {r.part_run.var() / (r.part_run.var() + r.part_wkt.var()):.0%}.")
cols_b = ["display_name", "role", "current_team", "n_net", "rank", "post_mean", "low80", "high80", "p_top10", "true_sr_pressure", "raw_run", "raw_wkt", "post_u", "availability_guard"]
cols_w = ["display_name", "role", "current_team", "n_net", "rank", "post_mean", "low80", "high80", "p_top10", "true_econ_pressure", "raw_run", "raw_wkt", "post_u", "availability_guard"]
print("\nTOP 15 BATTERS (provisional):");  display(rated_tables["bat"][cols_b].head(15).round(2))
print("TOP 15 BOWLERS (provisional):");   display(rated_tables["bowl"][cols_w].head(15).round(2))

players  corr with 2024-26 pressure  band low (5%)  band high (95%)
side    estimator (from 2022-23)                                                                                
batters pressure record only (raw)                67                      -0.090         -0.310            0.110
        one-part estimator (cell 5)               67                       0.000         -0.200            0.230
        two-part estimator (runs + wickets)       67                      -0.020         -0.210            0.170
        everyday level only (two-part)            67                      -0.020         -0.250            0.160
bowlers pressure record only (raw)                77                       0.060         -0.110            0.200
        one-part estimator (cell 5)               77                       0.270          0.120            0.430
        two-part estimator (runs + wickets)       77                       0.220          0.060            0.370
        everyday level only (two-part)            77                       0.290          0.150            0.420

BATTERS: league strike rate in batter-pressure balls = 145.2
  130 players rated. Rating spread across players: sd 9.58; typical uncertainty per player: sd 12.00. Share of the rating spread coming from the RUN part: 62%.
BOWLERS: league economy in bowler-pressure balls = 8.52 runs per over
  139 players rated. Rating spread across players: sd 1.48; typical uncertainty per player: sd 1.60. Share of the rating spread coming from the RUN part: 99%.

TOP 15 BATTERS (provisional):


,display_name,role,current_team,n_net,rank,post_mean,low80,high80,p_top10,true_sr_pressure,raw_run,raw_wkt,post_u,availability_guard
player_id,,,,,,,,,,,,,,
50284,Vaibhav Sooryavanshi,batter,Rajasthan Royals,115,1,41.800,26.130,57.480,0.920,185.430,50.140,6.070,40.750,OK
6396,Tim David,batter,Royal Challengers Bengaluru,182,2,28.490,14.320,42.660,0.620,170.450,46.400,0.850,25.690,OK
1699,Suryakumar Yadav,batter,Mumbai Indians,353,3,28.340,16.250,40.430,0.640,163.220,20.500,22.130,20.610,OK
5039,Rajat Patidar,batter,Royal Challengers Bengaluru,66,4,21.380,5.500,37.260,0.390,162.120,28.050,1.410,22.240,OK
768,Reeza Hendricks,batter,Not in IPL 2026,195,5,20.900,6.460,35.330,0.350,147.720,20.650,48.770,6.280,OK
5553,Phil Salt,batter,Royal Challengers Bengaluru,202,6,20.800,7.110,34.500,0.360,162.500,25.890,4.650,19.510,OK
1318,Andre Russell,batting all-rounder,Not in IPL 2026,90,7,19.510,3.670,35.350,0.330,163.300,23.420,1.230,20.350,EXCLUDED
4260,Heinrich Klaasen,batter,Sunrisers Hyderabad,126,8,18.800,4.190,33.410,0.290,156.600,26.510,12.390,16.530,OK
25669,Priyansh Arya,batter,Punjab Kings,73,9,18.040,1.360,34.710,0.290,170.400,55.730,-8.200,16.690,OK


TOP 15 BOWLERS (provisional):


,display_name,role,current_team,n_net,rank,post_mean,low80,high80,p_top10,true_econ_pressure,raw_run,raw_wkt,post_u,availability_guard
player_id,,,,,,,,,,,,,,
2933,Jasprit Bumrah,bowler,Mumbai Indians,217,1,5.790,3.990,7.600,0.960,7.000,5.650,-6.870,5.850,OK
781,Sunil Narine,bowling all-rounder,Kolkata Knight Riders,205,2,4.100,2.250,5.950,0.720,7.430,6.950,-1.090,3.650,OK
34485,Sufiyan Muqeem,bowler,Not in IPL 2026,80,3,3.210,0.980,5.440,0.470,7.770,13.610,-1.820,2.500,OK
7920,Nathan Ellis,bowler,Chennai Super Kings,164,4,2.990,0.990,4.990,0.440,7.770,5.150,2.400,2.680,OK
8661,Matheesha Pathirana,bowler,Kolkata Knight Riders,174,5,2.930,1.010,4.840,0.410,7.830,3.920,4.130,2.750,OK
5979,Abrar Ahmed,bowler,Not in IPL 2026,131,6,2.710,0.640,4.790,0.340,7.850,10.980,4.910,1.800,OK
360,Bhuvneshwar Kumar,bowler,Royal Challengers Bengaluru,333,7,2.660,0.980,4.350,0.320,7.850,2.370,2.100,2.730,OK
4722,Rashid Khan,bowling all-rounder,Gujarat Titans,362,8,2.650,1.010,4.280,0.300,7.830,3.670,-2.040,2.360,OK
2411,Sandeep Sharma,bowler,Rajasthan Royals,138,9,2.500,0.490,4.520,0.300,7.810,7.080,-3.060,2.010,OK


**Procedure**
1. Estimated each player's true pressure level in two parts, runs and wickets, each with its own reliability (the same two-level estimator as cell 5). The two parts are added into one rating, in net runs above expected per 100 pressure balls for batters and per 24 pressure balls for bowlers (higher is better for both).
2. Checked, on the 2022–23 → 2024–26 test, whether the two-part estimator predicts better than the one-part version.
3. Filtered to players with 60+ pressure balls whose role fits their workload.
4. Gave every rating a margin of error and simulated it 4,000 times to get the chance of landing in the top 10 and top 20.
5. Converted the run part into a "true strike rate" (batters) and "true economy" (bowlers) under pressure, against the league average in the pressure set.

**Observations**
- League strike rate in batter-pressure balls is 145.5, and league economy in bowler-pressure balls is 8.51 runs per over. 132 batters and 142 bowlers are rated.
- **Batters:** Vaibhav Sooryavanshi is the only clear separation (rating 40.9, 80% range 28.0–53.8, 97% chance of the top 10, true strike rate 184.5). Ranks 2–10 are a cluster, not a strict order (ratings 25.1 down to 17.0, top-10 chance only 27–66%): Tim David, Muhammad Waseem, Suryakumar Yadav, Rajat Patidar, Phil Salt, Heinrich Klaasen, Andre Russell, Finn Allen and Abhishek Sharma.
- **Bowlers:** Jasprit Bumrah is the clear #1 (rating 5.29, 80% range 3.83–6.74, 99% top-10 chance, true economy 7.00). Behind him come Sunil Narine (3.24), Matheesha Pathirana (2.75), Mustafizur Rahman, Sufiyan Muqeem, Nathan Ellis, Blessing Muzarabani, Bhuvneshwar Kumar, Rashid Khan and Eshan Malinga. All of them are within about 1.4 runs per 24 balls of each other, and their top-10 chances are 27–72%.
- **The rating is mostly an everyday-form rating with a small pressure adjustment.** The rating stays within about 1–2 points of the everyday level (`post_u`) for almost everyone: for example Sooryavanshi 40.9 against 39.8, and Bumrah 5.29 against 5.78. Pressure-specific form moves a rating only a little, which matches the weak persistence found in cell 5.
- **The uncertainty is as large as the spread.** The typical player's uncertainty is ±9.7 per 100 balls for batters and ±1.4 per 24 balls for bowlers, about the same as the spread of the ratings themselves (8.7 and 1.4). That is why the final lists must read as tiers, not exact ranks.
- **Where runs and wickets disagree, the run part drives the result.** The run part accounts for 72% of the batter rating spread and 96% of the bowler spread. Finn Allen shows the shrinkage at work: a raw wicket score of −56 per 100 balls on exactly 60 pressure balls, yet he stays ninth because the wicket noise is heavily discounted. Bumrah's raw wicket score is −9.9, with no effect on his rank.
- Several names are flagged for the final step. Andre Russell and Dinesh Karthik carry the EXCLUDED availability flag. Mark Wood, Abrar Ahmed, Sufiyan Muqeem, Waseem and Russell are "not in IPL 2026", so their auction availability is not settled. Sample size is thin for some top-15 names (Finn Allen 60 balls, Patidar 70, Arya 74, Karthik 91).

In [7]:
# ─────────────────────────────────────────────────────────────────────
# CELL 7 · SENSITIVITY: DO THE TOP LISTS SURVIVE OTHER REASONABLE CHOICES?
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL: every definition above involved a judgement (how strict "pressure" is, how many runs a wicket is worth, how comfortable the
# batting side must be). A list that changes completely when one judgement changes is not a finding. We re-run the whole rating under
# alternative choices and count how many of the main top-10 remain in the top 10 (and top 20), and how similar the full rankings are.
# TECHNICAL: Spearman rank correlation = correlation of the two rank lists (1 = identical order, 0 = unrelated).
from scipy.stats import spearmanr
def rating_for(side, flag, vmode="phase"):
    t = two_part(side, flag, vmode=vmode).join(names)
    role_ok = t.rated_as_batter.fillna(False) if side == "bat" else t.rated_as_bowler.fillna(False)
    return t[t.rated & role_ok & t.post_mean.notna()].post_mean

main = {s: rated_tables[s].post_mean for s in ["bat", "bowl"]}
variants = {}
for q in [15, 25, 30]:
    variants[f"threshold: top {q}% of leverage"] = {s: rating_for(s, f"{s}_press{q}") for s in ["bat", "bowl"]}
# alternative wicket value: one flat 15 runs per wicket for everyone (a common cricket rule of thumb) instead of the data-driven phase values
variants["wicket = flat 15 runs"] = {s: rating_for(s, f"{s}_press", vmode=15) for s in ["bat", "bowl"]}
rows = []
for name, v in variants.items():
    for s in ["bat", "bowl"]:
        a, b = main[s], v[s]; com = a.index.intersection(b.index)
        t10a, t10b = set(a.nlargest(10).index), set(b.nlargest(10).index); t20a, t20b = set(a.nlargest(20).index), set(b.nlargest(20).index)
        rows.append({"variant": name, "list": "batters" if s == "bat" else "bowlers", "players rated in both": len(com), "top-10 kept": len(t10a & t10b),
                     "top-20 kept": len(t20a & t20b), "rank similarity (Spearman)": spearmanr(a[com], b[com])[0]})
sens = pd.DataFrame(rows).set_index(["list", "variant"]).sort_index(); display(sens.round(2))
# players that appear in the top 10 of the main list AND in every variant: the most robust names
for s, lab in [("bat", "batters"), ("bowl", "bowlers")]:
    sets = [set(v[s].nlargest(10).index) for v in variants.values()]; base = set(main[s].nlargest(10).index)
    robust = base.intersection(*sets)
    print(f"{lab}: {len(robust)} of the main top 10 stay in the top 10 under ALL variants: {', '.join(pm.set_index('player_id').display_name.reindex(sorted(robust)).tolist())}")
    print(f"        in the top 10 of the main list but dropped by at least one variant: {', '.join(pm.set_index('player_id').display_name.reindex(sorted(base - robust)).tolist())}")

players rated in both  top-10 kept  top-20 kept  rank similarity (Spearman)
list    variant                                                                                                    
batters threshold: top 15% of leverage                    100            7           12                       0.890
        threshold: top 25% of leverage                    130            8           13                       0.870
        threshold: top 30% of leverage                    130            7           12                       0.820
        wicket = flat 15 runs                             130            8           13                       0.890
bowlers threshold: top 15% of leverage                    113            7           18                       0.940
        threshold: top 25% of leverage                    139            8           17                       0.960
        threshold: top 30% of leverage                    139            8           15                       0.920
        wicket = flat 15 runs                             139            8           19                       0.990

batters: 6 of the main top 10 stay in the top 10 under ALL variants: Andre Russell, Suryakumar Yadav, Phil Salt, Tim David, Muhammad Waseem, Vaibhav Sooryavanshi
        in the top 10 of the main list but dropped by at least one variant: Reeza Hendricks, Heinrich Klaasen, Rajat Patidar, Priyansh Arya
bowlers: 6 of the main top 10 stay in the top 10 under ALL variants: Bhuvneshwar Kumar, Sunil Narine, Jasprit Bumrah, Nathan Ellis, Matheesha Pathirana, Sufiyan Muqeem
        in the top 10 of the main list but dropped by at least one variant: Sandeep Sharma, Mustafizur Rahman, Rashid Khan, Abrar Ahmed


**Procedure**
1. Re-ran the full rating (the two-part estimator) under alternative choices: a stricter pressure threshold (top 15% of leverage), looser ones (top 25% and 30%), and a flat 15-run value for a wicket in place of the data-driven phase values.
2. For each variant, counted how many of the main top 10 and top 20 stay in, and measured the similarity of the whole ranking with the Spearman rank correlation (1 = identical order).
3. Listed the players who stay in the top 10 under every variant (the robust names) and those dropped by at least one.

**Observations**
- The batter list is stable. Top-10 overlap with the main list is 8–10 of 10 (the strictest 15% threshold costs two names), the top-20 overlap is 11–15, and rank similarity is 0.90–0.92.
- The bowler list is stable overall but more sensitive to how strict "pressure" is. The overlap is 9 / 8 / 8 of 10 for the 25% and 30% thresholds and the flat wicket value, but only 6 of 10 at the strict 15% threshold, which leaves a thinner pool (114 bowlers). Rank similarity is 0.95–0.98.
- Wicket value is not a driver. The flat 15-run alternative keeps 9 of 10 batters and 8 of 10 bowlers, so the conclusions do not hang on the data-driven phase values.
- Robust batters (top 10 in every variant): Vaibhav Sooryavanshi, Tim David, Suryakumar Yadav, Heinrich Klaasen, Phil Salt, Muhammad Waseem and Andre Russell. Dropped by at least one variant: Finn Allen, Rajat Patidar and Abhishek Sharma.
- Robust bowlers: Jasprit Bumrah, Sunil Narine, Matheesha Pathirana, Nathan Ellis and Sufiyan Muqeem. Dropped by at least one variant: Bhuvneshwar Kumar, Mustafizur Rahman, Rashid Khan, Blessing Muzarabani and Eshan Malinga.
- These are the names to carry forward as the most defensible. The rest of the top 10 is real but sits on the edge of the group, which fits the cluster and top-10 chances seen in cell 6.

In [8]:
# ─────────────────────────────────────────────────────────────────────
# CELL 7a · SAVE THE SENSITIVITY RANKS (so the final lists can show how stable each name is)
# ─────────────────────────────────────────────────────────────────────
# Each player's rank under the main definition and under each variant of cell 7 is stored. "worst_rank" is the lowest he ever falls; a player
# whose worst rank is still inside the top 10 is robust to the judgement calls we made.
rows = []
for s in ["bat", "bowl"]:
    cols = {"rank_main": main[s].rank(ascending=False)}
    for name, v in variants.items():
        key = "rank_" + name.replace("threshold: top ", "t").replace("% of leverage", "").replace("wicket = flat 15 runs", "flat15")
        cols[key] = v[s].rank(ascending=False)
    df = pd.DataFrame(cols); df["worst_rank"] = df.max(axis=1); df["variants_in_top10"] = (df.drop(columns="worst_rank") <= 10).sum(axis=1)
    df.insert(0, "side", s); df.index.name = "player_id"; rows.append(df.reset_index())
sens_out = pd.concat(rows, ignore_index=True)
print("Saved pressure_sensitivity:", sens_out.shape, save_table(sens_out, "pressure_sensitivity"))
display(sens_out[sens_out.rank_main <= 12].sort_values(["side", "rank_main"]).round(0).head(24))

Saved pressure_sensitivity: (355, 9) parquet


,player_id,side,rank_main,rank_t15,rank_t25,rank_t30,rank_flat15,worst_rank,variants_in_top10
168,50284,bat,1.000,1.000,1.000,1.000,1.000,1.000,5
121,6396,bat,2.000,2.000,2.000,3.000,2.000,3.000,5
38,1699,bat,3.000,3.000,5.000,7.000,3.000,7.000,5
109,5039,bat,4.000,NaN,4.000,4.000,10.000,10.000,4
16,768,bat,5.000,35.000,56.000,66.000,24.000,66.000,1
110,5553,bat,6.000,7.000,8.000,8.000,5.000,8.000,5
32,1318,bat,7.000,6.000,7.000,6.000,6.000,7.000,5
88,4260,bat,8.000,8.000,11.000,11.000,4.000,11.000,3
161,25669,bat,9.000,NaN,9.000,12.000,18.000,18.000,2
155,22565,bat,10.000,4.000,3.000,2.000,8.000,10.000,5


**Procedure**
1. For every rated player, saved his rank under the main definition and under each sensitivity variant of cell 7: the top 15%, 25% and 30% leverage thresholds, and a flat 15-run wicket value.
2. Added `worst_rank` (the lowest he falls in any variant) and `variants_in_top10` (in how many of the five rankings he is in the top 10, counting the main one). A blank rank means he was not rated in that variant, because the stricter 15% threshold leaves him under 60 pressure balls.

**Observations**
- **Top batters.** Six batters are in the top 10 in all five rankings: Sooryavanshi (worst rank 1), Tim David (2), Waseem (5), Suryakumar Yadav (6), Salt (10) and Klaasen (9). Russell stays in the top 10 in every ranking too (worst rank 8) but is removed later by the availability guard.
- **Batters that move.** Patidar and Finn Allen drop out of the strict 15% ranking entirely (fewer than 60 pressure balls under that definition). Abhishek Sharma (main rank 10) falls to 26 with the flat wicket value. Priyansh Arya (main rank 11) is never in the top 10 under any variant, and falls to 28 with the flat wicket value.
- **Top bowlers.** Five bowlers are in the top 10 in all five rankings: Bumrah (worst rank 1), Narine (3), Pathirana (4), Muqeem (5) and Ellis (8).
- **Bowlers that move.** Mustafizur Rahman (main rank 4) falls to 15 under the strict threshold and sits in the top 10 in only two rankings. Bhuvneshwar Kumar (worst rank 13), Rashid Khan (18), Malinga (14) and Muzarabani (13) can drop outside the top 10 under a different judgement call.
- The robust names are therefore the ones to trust most, and the "worst rank" column in the final lists shows this openly.

In [9]:
# ─────────────────────────────────────────────────────────────────────
# CELL 8 · SAVE (player rating tables, per-ball net columns, constants for the website)
# ─────────────────────────────────────────────────────────────────────
import json
for side, fn in [("bat", "pressure_batters"), ("bowl", "pressure_bowlers")]:
    out = rated_tables[side].reset_index()
    print(f"Saved {fn}: {out.shape}", save_table(out, fn))
# full tables of every player with pressure balls (not only rated), for the website's search box
for side, t, fn in [("bat", bat, "pressure_batters_all"), ("bowl", bowl, "pressure_bowlers_all")]:
    print(f"Saved {fn}: {t.reset_index().shape}", save_table(t.reset_index(), fn))
consts = {"wicket_value_batter_by_phase": V_BAT_PH, "wicket_value_bowler_by_phase": V_BOWL_PH, "scale": SCALE,
          "pressure_threshold_top_share": float(PRIMARY_Q) if "PRIMARY_Q" in globals() else 0.20, "comfort": 0.60,
          "hier": {s: {k: float(v) for k, v in two_part(s, f"{s}_press").attrs.items()} for s in ["bat", "bowl"]}}
(OUT_DIR / "models" / "pressure_constants.json").write_text(json.dumps(consts, indent=1))
ball = ball.drop(columns=["big_match"], errors="ignore")
print("Saved ball_master:", ball.shape, save_table(ball, "ball_master"))

Saved pressure_batters: (130, 51) parquet
Saved pressure_bowlers: (139, 51) parquet
Saved pressure_batters_all: (651, 37) parquet
Saved pressure_bowlers_all: (485, 37) parquet
Saved ball_master: (200961, 151) parquet


**Procedure**
1. Saved the two rating tables (batters and bowlers) with every column needed for lists and the website.
2. Saved full tables for every player with any pressure ball, for the website's search box.
3. Saved the modelling constants (wicket values by phase, scales, threshold, comfort rule, the two-part estimator's spreads) as a small JSON file.
4. Saved `ball_master` with the new per-ball columns (streaks, expected dots, net scores, late-pressure flag).

**Observations**
- Saved `pressure_batters` (132 rows, 51 columns), `pressure_bowlers` (142, 51), `pressure_batters_all` (644, 37) and `pressure_bowlers_all` (488, 37), all as parquet.
- `ball_master` is saved at 200,961 rows and 151 columns.
- Notebook 05c is complete. Notebook 06 can read the rating tables directly.